In [ ]:
# ============================================
# Arranque rápido: clonar repo si no existe
# ============================================
import os

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_LOCAL):
    print(f"El repo no está clonado. Clonando...")
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_LOCAL}
    !git config --global user.email "srcasares2@gmail.com"
    !git config --global user.name "{GITHUB_USER}"
    print("✅ Repo clonado")
else:
    %cd {REPO_LOCAL}
    print(f"✅ Repo encontrado: {REPO_LOCAL}")

!pwd
!git status

# 01 — Verificación del dataset CIC-IDS2017

**Proyecto:** VIGÍA (Sentry) — TFC
**Objetivo:** Verificar los 8 CSV en Drive, calcular checksums SHA256 y documentar.

**Requisito:** haber ejecutado antes `00_setup.ipynb`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_PATH = "/content/drive/MyDrive/cic-ids2017"

print(f"Buscando en: {DRIVE_PATH}\n")

if not os.path.exists(DRIVE_PATH):
    print(f"❌ La carpeta {DRIVE_PATH} NO existe.")
    print("Verifica el nombre exacto en Drive.")
else:
    archivos = os.listdir(DRIVE_PATH)
    csvs = [f for f in archivos if f.endswith(".csv")]

    print(f"Archivos encontrados: {len(csvs)} CSV\n")

    for f in sorted(csvs):
        ruta = os.path.join(DRIVE_PATH, f)
        tamaño_mb = os.path.getsize(ruta) / (1024 * 1024)
        print(f"  ✅ {f} ({tamaño_mb:.1f} MB)")

    if len(csvs) == 8:
        print("\n🎉 ¡Los 8 CSV están listos!")
    elif len(csvs) < 8:
        print(f"\n⚠️ Faltan {8 - len(csvs)} archivos.")

In [ ]:
# ============================================
# CONFIGURACIÓN
# ============================================

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"
DRIVE_PATH  = "/content/drive/MyDrive/cic-ids2017"

print(f"Repositorio: {GITHUB_USER}/{REPO_NAME}")
print(f"Carpeta del dataset: {DRIVE_PATH}")

## 1. Verificar que el repositorio está clonado

Si no está clonado, hay que ejecutar `00_setup.ipynb` primero.

In [ ]:
import os

if not os.path.exists(REPO_LOCAL):
    print(f"❌ El repo no está clonado en {REPO_LOCAL}")
    print("   Ejecuta primero notebooks/00_setupsentry.ipynb")
else:
    %cd {REPO_LOCAL}
    print(f"✅ Repo encontrado: {REPO_LOCAL}")
    !pwd
    !git status

## 4. Calcular checksums SHA256

Cada archivo se verifica con su hash SHA256. Si un solo bit cambia, el hash cambia.

**Tiempo estimado:** 1-2 minutos.

In [ ]:
import hashlib
import os

def calcular_sha256(ruta, bloque=65536):
    hash_sha = hashlib.sha256()
    with open(ruta, "rb") as f:
        for chunk in iter(lambda: f.read(bloque), b""):
            hash_sha.update(chunk)
    return hash_sha.hexdigest()

resultados = []
for f in csvs:
    ruta = os.path.join(DRIVE_PATH, f)
    print(f"Calculando {f}...", end=" ")
    hash_val = calcular_sha256(ruta)
    resultados.append((f, hash_val))
    print("OK")

print(f"\n✅ Checksums calculados para {len(resultados)} archivos.")

## 5. Guardar checksums en el repositorio

Guardamos los hashes en `docs/checksums.txt` para trazabilidad.

In [ ]:
%cd {REPO_LOCAL}

from datetime import datetime

with open("docs/checksums.txt", "w") as f:
    f.write("# Checksums SHA256 del dataset CIC-IDS2017\n")
    f.write(f"# Calculado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("# Fuente: https://www.unb.ca/cic/datasets/ids-2017.html\n")
    f.write("#\n")
    f.write("# Para verificar de nuevo:\n")
    f.write("#   sha256sum archivo.csv\n")
    f.write("\n")
    for nombre, hash_val in resultados:
        f.write(f"{hash_val}  {nombre}\n")

print("=== docs/checksums.txt ===")
!cat docs/checksums.txt

## 6. Actualizar diario de decisiones

In [ ]:
%cd {REPO_LOCAL}

from datetime import datetime

fecha = datetime.now().strftime("%Y-%m-%d")

entrada = f"""

---

## {fecha} — Verificación del dataset CIC-IDS2017

**Qué probé:**
- Verificar los 8 CSV en Google Drive.
- Calcular checksums SHA256.
- Documentar en `docs/checksums.txt`.

**Qué funcionó:**
- Los 8 CSV están en `/content/drive/MyDrive/cic-ids2017/`.
- Checksums calculados correctamente.

**Archivos del dataset:**
"""
for nombre, _ in resultados:
    entrada += f"- {nombre}\n"

entrada += """
**Próximos pasos:**
- EDA (análisis exploratorio) del dataset.
- Distribución de clases, valores nulos, infinitos, negativos.
"""

with open("docs/diario.md", "a") as f:
    f.write(entrada)

print("=== Últimas líneas del diario ===")
!tail -30 docs/diario.md

## 7. Commit y push

**Genera un token en** https://github.com/settings/tokens (scope `repo`) **si no tienes uno**.

In [ ]:
from getpass import getpass

TOKEN = getpass("Pega tu token: ")
print(f"Token capturado: {len(TOKEN)} caracteres")

%cd {REPO_LOCAL}
!git remote set-url origin https://{TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git
!git remote get-url origin | grep -q "ghp_" && echo "✅ Token configurado"
!git push

## ✅ Día 2 completado

- [ ] 8 CSV verificados en Drive
- [ ] Checksums SHA256 calculados
- [ ] `docs/checksums.txt` creado
- [ ] `docs/diario.md` actualizado
- [ ] Push a GitHub hecho

**Verificar:** https://github.com/AguCS231/sentry/tree/main/docs

In [ ]:
%cd /content/sentry

!git add .
!git commit -m "Añadir checksums SHA256 del dataset CIC-IDS2017"

In [ ]:
from getpass import getpass

TOKEN = getpass("Pega tu token: ")
print(f"Token capturado: {len(TOKEN)} caracteres (esperado: 40)")

%cd /content/sentry
!git remote set-url origin https://{TOKEN}@github.com/AguCS231/sentry.git
!git remote get-url origin | grep -q "ghp_" && echo "✅ Token configurado"
!git push